## import packages

In [ ]:
from pyflamegpu import *
import pyflamegpu.codegen
import sys
import math

## Define the FLAME GPU model name

In [ ]:
# Define the FLAME GPU model: 这个可以在后续的可视化窗口改名字
model = pyflamegpu.ModelDescription("Social_physical_shelter_Opt")

## host function for env

### for familirity map

In [ ]:
# Define an host function called write_env_hostfn
class write_env_hostfn(pyflamegpu.HostFunction):
  
  def __init__(self):
    super().__init__()  
  
  def run(self,FLAMEGPU):

      # Retrieve the environment macro property bar of type int array[5][5]

      # Update some of the values
      # foo = 12.0; is not allowed
      FLAMEGPU.environment.importMacroProperty("map", "attraction_matrix_campus.json");

      FLAMEGPU.environment.exportMacroProperty("map", "out1.json");

    # Python does not allow the increment operator to be overridden

### for graph

In [ ]:
# Define an host function called directed_graph_hostfn
class directed_graph_hostfn(pyflamegpu.HostFunction):
  def run(self,FLAMEGPU):
    # Fetch a handle to the directed graph
    fgraph = FLAMEGPU.environment.getDirectedGraph("fgraph")
    # Import a different graph
    fgraph.importGraph("expanded_visibility_graph.json");

## messages

In [ ]:
# 可以获取一定距离内的消息
message = model.newMessageSpatial3D("location")
# Configure the message list
message.setMin(0, 0,0)
message.setMax(ENV_WIDTH, ENV_WIDTH,ENV_WIDTH)
message.setRadius(2)
# Add extra variables to the message
# X Y (Z) are implicit for spatial messages
message.newVariableID("id")

In [ ]:
stairwell_message = model.newMessageSpatial3D("location_stairwell")
stairwell_message.setMin(0, 0,0)
stairwell_message.setMax(500, 500, 500)
message.setRadius(200)
stairwell_message.newVariableID("id")
stairwell_message.newVariableFloat("class")

## agents set variables

### student agents

In [ ]:
# Assign the agent some variables (ID is implicit to agents, so we don't define it ourselves)
student_agent = model.newAgent("student_agent")
student_agent.newVariableFloat("x")
student_agent.newVariableFloat("y")
student_agent.newVariableInt("building_id")
student_agent.newVariableInt("point_id")
student_agent.newVariableFloat("z")
student_agent.newVariableFloat("drift", 0)
#set the states for student agents
student_agent.newState("not evacuate")
student_agent.newState("focused")
student_agent.newState("building evacuate")
student_agent.newState("stairwell evacuate")
student_agent.newState("neighborhood evacuate")



### stairwell agents

In [ ]:
stairwell_agent = model.newAgent("stairwell_agent")
stairwell_agent.newVariableFloat("x")
stairwell_agent.newVariableFloat("y")
stairwell_agent.newVariableInt("stairwell_id")
stairwell_agent.newVariableFloat("z")

## environment

In [ ]:
# Fetch the model's environment
env = model.Environment()

### familiarity map

In [ ]:
env.newMacroPropertyInt("map", 100, 80)

### road planning graph

In [ ]:
# Declare a new directed graph named 'fgraph'
fgraph = env.newDirectedGraph("fgraph")
# Attach an float[2] property 'bar' to vertices
fgraph.newVertexPropertyArrayFloat("bar", 2)
# Attach an int property 'foo' to edges
fgraph.newEdgePropertyFloat("foo")

## agent functions

## function write in

In [ ]:
# translate the agent functions from Python to C++
#output_func_translated = pyflamegpu.codegen.translate(output_message)
#input_func_translated = pyflamegpu.codegen.translate(input_message)

move_func_translated= pyflamegpu.codegen.translate(move)
move_fn = student_agent.newRTCFunction("move",move_func_translated)

move_s_fn = stairwell_agent.newRTCFunction("move",move_func_translated)

move_s_fn.dependsOn(move_fn)


## model simulation write in

In [ ]:
# Specify the desired StepLoggingConfig
step_log_cfg = pyflamegpu.StepLoggingConfig(model)
# Log every step
step_log_cfg.setFrequency(1)
# Include the mean of the "point" agent population's variable 'drift'
step_log_cfg.agent("point").logMeanFloat("foo")

# Create and init the simulation
cuda_model = pyflamegpu.CUDASimulation(model)

## agent initialization

In [ ]:
# 导入flamegpu_init_code.py中的初始化函数
import sys 
import os
sys.path.append('data/output')
from flamegpu_init_code import initialize_student_agent_population

# 添加学生代理类型
# 基于data/output/flamegpu_init_code.py的学生代理初始化

# 初始化学生代理种群
initialize_student_agent_population(model, cuda_model)

# Specify the desired StepLoggingConfig
step_log_cfg = pyflamegpu.StepLoggingConfig(model)
# Log every step
step_log_cfg.setFrequency(1) 
# Include the mean of the "point" agent population's variable 'drift'
step_log_cfg.agent("student_agent").logMeanFloat("drift")
step_log_cfg.agent("student_agent").logMeanFloat("x")
step_log_cfg.agent("student_agent").logMeanFloat("y")


cuda_model.initialise(sys.argv)


# Attach the logging config
cuda_model.setStepLog(step_log_cfg) 

## visualization part

In [ ]:
# Only run this block if pyflamegpu was built with visualisation support
if pyflamegpu.VISUALISATION:
    # Create visualisation
    m_vis = cuda_model.getVisualisation()
    # Set the initial camera location and speed

    m_vis.setInitialCameraTarget(270, 205, 0)
    m_vis.setInitialCameraLocation(240, 100, 100)
    m_vis.setCameraSpeed(0.01)
    m_vis.setSimulationSpeed(25)
    # Add "point" agents to the visualisation

    
    # Add "student_agent" agents to the visualisation
    student_agt = m_vis.addAgent("student_agent")
    student_agt.setModel(pyflamegpu.ICOSPHERE);
    student_agt.setModelScale(1/1.0);
    # Mark the environment bounds.

    stairwell_agt = m_vis.addAgent("stairwell_agent")
    stairwell_agt.setModel(pyflamegpu.ICOSPHERE);
    stairwell_agt.setModelScale(1/0.5);
    stairwell_agt.setColor(pyflamegpu.RED);
     
    pen = m_vis.newPolylineSketch(1, 1, 1, 0.2)
    pen.addVertex(275, 637, 0) # 起始点
    pen.addVertex(69, 510, 0)
    pen.addVertex(0, 301, 0)
    pen.addVertex(1, 167, 0)
    pen.addVertex(29, 142, 0)
    pen.addVertex(57, 98, 0)
    pen.addVertex(118, 67, 0)
    pen.addVertex(109, 24, 0)
    pen.addVertex(287, 0, 0)
    pen.addVertex(286, 45, 0)
    pen.addVertex(405, 154, 0)
    pen.addVertex(435, 131, 0)
    pen.addVertex(436, 72, 0)
    pen.addVertex(467, 41, 0)
    pen.addVertex(501, 35, 0)
    pen.addVertex(543, 47, 0)
    pen.addVertex(275, 637, 0) # 闭合点
    # Open the visualiser window 
    m_vis.activate()

# Run the simulation
cuda_model.simulate()

if pyflamegpu.VISUALISATION:
    # Keep the visualisation window active after the simulation has completed
    m_vis.join()


# python src/test_3d_with_function.py -s 10 --out-step step.json